### In order to properly classify products into XYZ and FSN classifications, we need atleast 4 months of sales data. Hence, we need to perform the necessary filtering for classification.

In [12]:
from pathlib import Path
import pandas as pd


DATA_DIR = Path("archive")
DAILY_OUTPUT_FILE = Path("product_daily_sales_timeseries.csv")
MONTHLY_OUTPUT_FILE = Path("product_monthly_sales_timeseries.csv")


def load_required_csv(file_name: str, **read_kwargs) -> pd.DataFrame:
    file_path = DATA_DIR / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required file: {file_path}")
    return pd.read_csv(file_path, **read_kwargs)


def mode_or_unknown(series: pd.Series) -> str:
    non_null = series.dropna()
    if non_null.empty:
        return "Unknown"
    modes = non_null.mode()
    if modes.empty:
        return str(non_null.iloc[0])
    return str(modes.iloc[0])


def build_product_sales_timeseries() -> tuple[pd.DataFrame, pd.DataFrame]:
    orders = load_required_csv(
        "olist_orders_dataset.csv",
        usecols=["order_id", "order_status", "order_purchase_timestamp"],
    )
    order_items = load_required_csv(
        "olist_order_items_dataset.csv",
        usecols=["order_id", "order_item_id", "product_id", "price", "freight_value"],
    )
    products = load_required_csv(
        "olist_products_dataset.csv",
        usecols=["product_id", "product_category_name"],
    )
    category_translation = load_required_csv("product_category_name_translation.csv")

    orders["order_purchase_timestamp"] = pd.to_datetime(
        orders["order_purchase_timestamp"], errors="coerce"
    )
    orders = orders[orders["order_purchase_timestamp"].notna()].copy()

    excluded_status = ["canceled", "unavailable"]
    orders["order_status"] = orders["order_status"].astype("string").str.strip().str.lower()
    orders = orders[~orders["order_status"].isin(excluded_status)].copy()

    products_enriched = products.merge(
        category_translation,
        on="product_category_name",
        how="left",
    )
    products_enriched["product_category_name"] = (
        products_enriched["product_category_name"].astype("string").fillna("unknown")
    )
    products_enriched["product_category_name_english"] = (
        products_enriched["product_category_name_english"].astype("string").fillna("unknown")
    )

    merged = order_items.merge(orders, on="order_id", how="inner")
    merged = merged.merge(products_enriched, on="product_id", how="left")

    merged["order_date"] = merged["order_purchase_timestamp"].dt.floor("D")
    merged["line_sales_value"] = (
        pd.to_numeric(merged["price"], errors="coerce").fillna(0)
        + pd.to_numeric(merged["freight_value"], errors="coerce").fillna(0)
    )

    product_category_map = (
        merged.groupby("product_id", as_index=False)
        .agg(
            product_category_name=("product_category_name", mode_or_unknown),
            product_category_name_english=("product_category_name_english", mode_or_unknown),
        )
        .copy()
    )

    daily_sales = (
        merged.groupby(["product_id", "order_date"], as_index=False)
        .agg(
            quantity_sold=("order_item_id", "count"),
            unique_orders=("order_id", "nunique"),
            gross_item_sales=("price", "sum"),
            gross_freight=("freight_value", "sum"),
            total_sales_value=("line_sales_value", "sum"),
            avg_unit_price=("price", "mean"),
        )
        .merge(product_category_map, on="product_id", how="left")
        .sort_values(["product_id", "order_date"])
        .reset_index(drop=True)
    )

    daily_sales["order_month"] = daily_sales["order_date"].dt.to_period("M").dt.to_timestamp()
    monthly_sales = (
        daily_sales.groupby(["product_id", "order_month"], as_index=False)
        .agg(
            quantity_sold=("quantity_sold", "sum"),
            unique_orders=("unique_orders", "sum"),
            gross_item_sales=("gross_item_sales", "sum"),
            gross_freight=("gross_freight", "sum"),
            total_sales_value=("total_sales_value", "sum"),
            avg_unit_price=("avg_unit_price", "mean"),
            product_category_name=("product_category_name", mode_or_unknown),
            product_category_name_english=("product_category_name_english", mode_or_unknown),
        )
        .sort_values(["product_id", "order_month"])
        .reset_index(drop=True)
    )

    month_coverage = (
        monthly_sales.groupby("product_id", as_index=False)["order_month"]
        .nunique()
        .rename(columns={"order_month": "months_available"})
    )
    eligible_products = set(
        month_coverage.loc[month_coverage["months_available"] >= 4, "product_id"]
    )

    daily_sales = (
        daily_sales[daily_sales["product_id"].isin(eligible_products)]
        .sort_values(["product_id", "order_date"])
        .reset_index(drop=True)
    )
    monthly_sales = (
        monthly_sales[monthly_sales["product_id"].isin(eligible_products)]
        .sort_values(["product_id", "order_month"])
        .reset_index(drop=True)
    )

    return daily_sales, monthly_sales


def main() -> None:
    daily_sales, monthly_sales = build_product_sales_timeseries()
    daily_sales = daily_sales.sort_values(["product_id", "order_date"]).reset_index(drop=True)
    monthly_sales = monthly_sales.sort_values(["product_id", "order_month"]).reset_index(drop=True)

    daily_sales.to_csv(DAILY_OUTPUT_FILE, index=False)
    monthly_sales.to_csv(MONTHLY_OUTPUT_FILE, index=False)

    min_months_per_product = (
        monthly_sales.groupby("product_id")["order_month"].nunique().min()
        if not monthly_sales.empty
        else 0
    )

    print("Daily product time-series shape:", daily_sales.shape)
    print("Monthly product time-series shape:", monthly_sales.shape)
    print("Unique products:", daily_sales["product_id"].nunique())
    print("Minimum months per product:", int(min_months_per_product))
    print(
        "Date range:",
        daily_sales["order_date"].min().date(),
        "to",
        daily_sales["order_date"].max().date(),
    )
    print("Saved:", DAILY_OUTPUT_FILE)
    print("Saved:", MONTHLY_OUTPUT_FILE)

    display(daily_sales.head())


main()

Daily product time-series shape: (47657, 11)
Monthly product time-series shape: (22222, 10)
Unique products: 3685
Minimum months per product: 4
Date range: 2016-09-04 to 2018-09-03
Saved: product_daily_sales_timeseries.csv
Saved: product_monthly_sales_timeseries.csv


,product_id,order_date,quantity_sold,unique_orders,gross_item_sales,gross_freight,total_sales_value,avg_unit_price,product_category_name,product_category_name_english,order_month
0,001b72dfd63e9833e8c02742adf472e3,2017-02-15,2,1,69.98,32.10,102.08,34.99,moveis_decoracao,furniture_decor,2017-02-01
1,001b72dfd63e9833e8c02742adf472e3,2017-02-22,1,1,34.99,14.11,49.10,34.99,moveis_decoracao,furniture_decor,2017-02-01
2,001b72dfd63e9833e8c02742adf472e3,2017-03-01,1,1,34.99,17.78,52.77,34.99,moveis_decoracao,furniture_decor,2017-03-01
3,001b72dfd63e9833e8c02742adf472e3,2017-07-02,1,1,34.99,15.10,50.09,34.99,moveis_decoracao,furniture_decor,2017-07-01
4,001b72dfd63e9833e8c02742adf472e3,2017-07-08,2,1,69.98,15.56,85.54,34.99,moveis_decoracao,furniture_decor,2017-07-01


In [16]:
from pathlib import Path
import pandas as pd


DAILY_INPUT_FILE = Path("product_daily_sales_timeseries.csv")
PRODUCT_CLASS_OUTPUT_FILE = Path("product_class_mapping.csv")


def safe_cv(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return 0.0
    mean_val = values.mean()
    if mean_val == 0:
        return 0.0
    return float(values.std(ddof=0) / mean_val)


def classify_by_percentile_rank(series: pd.Series, labels: list[str]) -> pd.Series:
    ranked = series.rank(method="first", pct=True)
    bins = [0.0, 1 / 3, 2 / 3, 1.0]
    categorized = pd.cut(
        ranked,
        bins=bins,
        labels=labels,
        include_lowest=True,
    )
    return categorized.astype("string")


def classify_abc_by_cumulative_value(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    sorted_values = values.sort_values(ascending=False)
    total_value = float(sorted_values.sum())

    if total_value <= 0:
        return pd.Series("C", index=values.index, dtype="string")

    cumulative_before = sorted_values.cumsum().shift(fill_value=0) / total_value
    abc_sorted = pd.Series("C", index=sorted_values.index, dtype="string")
    abc_sorted.loc[cumulative_before < 0.80] = "A"
    abc_sorted.loc[(cumulative_before >= 0.80) & (cumulative_before < 0.95)] = "B"

    return abc_sorted.reindex(values.index).fillna("C").astype("string")


daily_df = pd.read_csv(DAILY_INPUT_FILE)

if "gross_item_sales" in daily_df.columns:
    daily_df["sales_value_for_abc"] = pd.to_numeric(daily_df["gross_item_sales"], errors="coerce").fillna(0)
elif "total_sales_value" in daily_df.columns:
    daily_df["sales_value_for_abc"] = pd.to_numeric(daily_df["total_sales_value"], errors="coerce").fillna(0)
else:
    quantity = pd.to_numeric(daily_df.get("quantity_sold", 0), errors="coerce").fillna(0)
    avg_price = pd.to_numeric(daily_df.get("avg_unit_price", 0), errors="coerce").fillna(0)
    daily_df["sales_value_for_abc"] = quantity * avg_price

cv_df = (
    daily_df.groupby("product_id", as_index=False)
    .agg(demand_cv=("quantity_sold", safe_cv))
    .copy()
)

stats_df = (
    daily_df.groupby("product_id", as_index=False)
    .agg(
        days_active=("order_date", "nunique"),
        total_quantity=("quantity_sold", "sum"),
        avg_daily_quantity=("quantity_sold", "mean"),
        total_sales_for_abc=("sales_value_for_abc", "sum"),
    )
    .merge(cv_df, on="product_id", how="left")
)

# XYZ: lower CV means more stable demand (X), higher CV means more variable demand (Z).
stats_df["xyz_class"] = classify_by_percentile_rank(
    stats_df["demand_cv"], labels=["X", "Y", "Z"]
)

# FSN: lower average movement is N, then S, highest movement is F.
stats_df["fsn_class"] = classify_by_percentile_rank(
    stats_df["avg_daily_quantity"], labels=["N", "S", "F"]
)

# ABC: high contribution products are A, medium B, low C.
stats_df["abc_class"] = classify_abc_by_cumulative_value(stats_df["total_sales_for_abc"])

stats_df["class"] = stats_df["fsn_class"] + stats_df["xyz_class"]
stats_df["is_demand_stable"] = (stats_df["xyz_class"] == "X").astype(int)
stats_df["is_fast_moving"] = (stats_df["fsn_class"] == "F").astype(int)

class_mapping = stats_df[
    [
        "product_id",
        "xyz_class",
        "fsn_class",
        "abc_class",
        "class",
        "is_demand_stable",
        "is_fast_moving",
        "demand_cv",
        "avg_daily_quantity",
        "days_active",
        "total_quantity",
        "total_sales_for_abc",
    ]
].copy()

class_mapping.to_csv(PRODUCT_CLASS_OUTPUT_FILE, index=False)

print("Product class mapping shape:", class_mapping.shape)
print("XYZ distribution:")
print(class_mapping["xyz_class"].value_counts().sort_index())
print("\nFSN distribution:")
print(class_mapping["fsn_class"].value_counts().sort_index())
print("\nABC distribution:")
print(class_mapping["abc_class"].value_counts().sort_index())
print("\nClassification source:", DAILY_INPUT_FILE)
print("Saved:", PRODUCT_CLASS_OUTPUT_FILE)

display(class_mapping.head())

Product class mapping shape: (1228, 12)
XYZ distribution:
xyz_class
X    409
Y    409
Z    410
Name: count, dtype: Int64

FSN distribution:
fsn_class
F    410
N    409
S    409
Name: count, dtype: Int64

ABC distribution:
abc_class
A    486
B    389
C    353
Name: count, dtype: Int64

Classification source: product_daily_sales_timeseries.csv
Saved: product_class_mapping.csv


,product_id,xyz_class,fsn_class,abc_class,class,is_demand_stable,is_fast_moving,demand_cv,avg_daily_quantity,days_active,total_quantity,total_sales_for_abc
0,005030ef108f58b46b78116f754d8d38,Y,S,C,SY,0,0,0.255125,1.083333,12,13,191.87
1,013e6676e0e3529e5909ff54370daddf,Z,F,A,FZ,0,1,0.306186,1.600000,5,8,2299.20
2,017692475c1c954ff597feda05131d73,Y,S,C,SY,0,0,0.255125,1.083333,24,26,366.65
3,02a0c276d5a252215b48269dea73a6e7,Z,F,B,FZ,0,1,0.306186,1.142857,7,8,840.00
4,02fbee632a2044d48ab16d57eec4db58,Z,F,B,FZ,0,1,0.293972,1.125000,8,9,779.20


In [17]:
from pathlib import Path
import pandas as pd


MONTHLY_INPUT_FILE = Path("product_monthly_sales_timeseries.csv")
DAILY_INPUT_FILE = Path("product_daily_sales_timeseries.csv")
PRODUCT_CLASS_INPUT_FILE = Path("product_class_mapping.csv")
MONTHLY_CLASSIFIED_OUTPUT_FILE = Path("product_monthly_sales_timeseries_classified.csv")
DAILY_CLASSIFIED_OUTPUT_FILE = Path("product_daily_sales_timeseries_classified.csv")


def add_class_columns(
    df: pd.DataFrame,
    class_map: pd.DataFrame,
) -> pd.DataFrame:
    columns_to_drop = ["class", "abc_class", "is_demand_stable", "is_fast_moving"]
    base_df = df.drop(
        columns=[col for col in columns_to_drop if col in df.columns],
        errors="ignore",
    )

    enriched_df = base_df.merge(class_map, on="product_id", how="left")
    enriched_df["class"] = enriched_df["class"].astype("string").fillna("NY")
    enriched_df["abc_class"] = enriched_df["abc_class"].astype("string").fillna("C")
    enriched_df["is_demand_stable"] = (
        pd.to_numeric(enriched_df["is_demand_stable"], errors="coerce").fillna(0).astype(int)
    )
    enriched_df["is_fast_moving"] = (
        pd.to_numeric(enriched_df["is_fast_moving"], errors="coerce").fillna(0).astype(int)
    )
    return enriched_df


class_mapping_df = pd.read_csv(
    PRODUCT_CLASS_INPUT_FILE,
    usecols=["product_id", "class", "abc_class", "is_demand_stable", "is_fast_moving"],
)

monthly_df = pd.read_csv(MONTHLY_INPUT_FILE)
daily_df = pd.read_csv(DAILY_INPUT_FILE)

monthly_classified = add_class_columns(monthly_df, class_mapping_df)
daily_classified = add_class_columns(daily_df, class_mapping_df)

monthly_classified.to_csv(MONTHLY_CLASSIFIED_OUTPUT_FILE, index=False)
# Update the original daily file as requested, and keep a separate classified copy.
daily_classified.to_csv(DAILY_INPUT_FILE, index=False)
daily_classified.to_csv(DAILY_CLASSIFIED_OUTPUT_FILE, index=False)

print("Classified monthly dataset shape:", monthly_classified.shape)
print("Classified daily dataset shape:", daily_classified.shape)
print("Rows without class in monthly:", monthly_classified["class"].isna().sum())
print("Rows without class in daily:", daily_classified["class"].isna().sum())
print("Rows without abc_class in monthly:", monthly_classified["abc_class"].isna().sum())
print("Rows without abc_class in daily:", daily_classified["abc_class"].isna().sum())
print("Saved:", MONTHLY_CLASSIFIED_OUTPUT_FILE)
print("Updated:", DAILY_INPUT_FILE)
print("Saved:", DAILY_CLASSIFIED_OUTPUT_FILE)

display(daily_classified.head())

Classified monthly dataset shape: (7863, 14)
Classified daily dataset shape: (14814, 15)
Rows without class in monthly: 0
Rows without class in daily: 0
Rows without abc_class in monthly: 0
Rows without abc_class in daily: 0
Saved: product_monthly_sales_timeseries_classified.csv
Updated: product_daily_sales_timeseries.csv
Saved: product_daily_sales_timeseries_classified.csv


,product_id,order_date,quantity_sold,unique_orders,gross_item_sales,gross_freight,total_sales_value,avg_unit_price,product_category_name,product_category_name_english,order_month,abc_class,class,is_demand_stable,is_fast_moving
0,005030ef108f58b46b78116f754d8d38,2017-10-20,1,1,13.99,15.09,29.08,13.99,perfumaria,perfumery,2017-10-01,C,SY,0,0
1,005030ef108f58b46b78116f754d8d38,2017-11-14,1,1,13.99,7.78,21.77,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0
2,005030ef108f58b46b78116f754d8d38,2017-11-16,1,1,13.99,7.78,21.77,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0
3,005030ef108f58b46b78116f754d8d38,2017-11-18,1,1,13.99,14.10,28.09,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0
4,005030ef108f58b46b78116f754d8d38,2017-11-24,1,1,13.99,14.10,28.09,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0


In [18]:
from pathlib import Path
import pandas as pd


MONTHLY_TS_FILE = Path("product_monthly_sales_timeseries.csv")
DAILY_TS_FILE = Path("product_daily_sales_timeseries.csv")
CLASS_MAP_FILE = Path("product_class_mapping.csv")
MONTHLY_OUTPUT_FILE = Path("product_monthly_sales.csv")
DAILY_OUTPUT_FILE = Path("product_daily_sales.csv")


monthly_df = pd.read_csv(MONTHLY_TS_FILE, parse_dates=["order_month"])
daily_df = pd.read_csv(DAILY_TS_FILE, parse_dates=["order_date"])
class_map_df = pd.read_csv(
    CLASS_MAP_FILE,
    usecols=["product_id", "class", "abc_class", "is_demand_stable", "is_fast_moving"],
)
class_map_df["class"] = class_map_df["class"].astype("string").fillna("NY")
class_map_df["abc_class"] = class_map_df["abc_class"].astype("string").fillna("C")
class_map_df["is_demand_stable"] = (
    pd.to_numeric(class_map_df["is_demand_stable"], errors="coerce").fillna(0).astype(int)
)
class_map_df["is_fast_moving"] = (
    pd.to_numeric(class_map_df["is_fast_moving"], errors="coerce").fillna(0).astype(int)
)

monthly_for_filter = monthly_df.drop(
    columns=[
        col
        for col in ["class", "abc_class", "is_demand_stable", "is_fast_moving"]
        if col in monthly_df.columns
    ],
    errors="ignore",
).merge(class_map_df, on="product_id", how="left")
monthly_for_filter["class"] = monthly_for_filter["class"].astype("string").fillna("NY")
monthly_for_filter["abc_class"] = monthly_for_filter["abc_class"].astype("string").fillna("C")
monthly_for_filter["is_demand_stable"] = (
    pd.to_numeric(monthly_for_filter["is_demand_stable"], errors="coerce").fillna(0).astype(int)
)
monthly_for_filter["is_fast_moving"] = (
    pd.to_numeric(monthly_for_filter["is_fast_moving"], errors="coerce").fillna(0).astype(int)
)

# Use monthly data as the source of truth for filtering decisions.
months_by_product = (
    monthly_for_filter.groupby("product_id", as_index=False)["order_month"]
    .nunique()
    .rename(columns={"order_month": "months_available"})
)
eligible_by_months = set(
    months_by_product.loc[months_by_product["months_available"] >= 4, "product_id"]
)

class_by_product = (
    monthly_for_filter.groupby("product_id", as_index=False)
    .agg(class_code=("class", "first"))
)
eligible_by_class = set(
    class_by_product.loc[
        ~class_by_product["class_code"].str.startswith("N", na=False)
        & ~class_by_product["class_code"].str.endswith("Z", na=False),
        "product_id",
    ]
)

eligible_product_ids = eligible_by_months & eligible_by_class

monthly_filtered_ts = (
    monthly_df[monthly_df["product_id"].isin(eligible_product_ids)]
    .sort_values(["product_id", "order_month"])
    .reset_index(drop=True)
)
daily_filtered_ts = (
    daily_df[daily_df["product_id"].isin(eligible_product_ids)]
    .sort_values(["product_id", "order_date"])
    .reset_index(drop=True)
)

monthly_filtered_output = monthly_filtered_ts.drop(
    columns=[
        col
        for col in ["class", "abc_class", "is_demand_stable", "is_fast_moving"]
        if col in monthly_filtered_ts.columns
    ],
    errors="ignore",
).merge(class_map_df, on="product_id", how="left")
monthly_filtered_output["class"] = monthly_filtered_output["class"].astype("string").fillna("NY")
monthly_filtered_output["abc_class"] = monthly_filtered_output["abc_class"].astype("string").fillna("C")
monthly_filtered_output["is_demand_stable"] = (
    pd.to_numeric(monthly_filtered_output["is_demand_stable"], errors="coerce").fillna(0).astype(int)
)
monthly_filtered_output["is_fast_moving"] = (
    pd.to_numeric(monthly_filtered_output["is_fast_moving"], errors="coerce").fillna(0).astype(int)
)

daily_filtered_output = daily_filtered_ts.drop(
    columns=[
        col
        for col in ["class", "abc_class", "is_demand_stable", "is_fast_moving"]
        if col in daily_filtered_ts.columns
    ],
    errors="ignore",
).merge(class_map_df, on="product_id", how="left")
daily_filtered_output["class"] = daily_filtered_output["class"].astype("string").fillna("NY")
daily_filtered_output["abc_class"] = daily_filtered_output["abc_class"].astype("string").fillna("C")
daily_filtered_output["is_demand_stable"] = (
    pd.to_numeric(daily_filtered_output["is_demand_stable"], errors="coerce").fillna(0).astype(int)
)
daily_filtered_output["is_fast_moving"] = (
    pd.to_numeric(daily_filtered_output["is_fast_moving"], errors="coerce").fillna(0).astype(int)
)

# Update the timeseries files.
monthly_filtered_ts.to_csv(MONTHLY_TS_FILE, index=False)
daily_filtered_ts.to_csv(DAILY_TS_FILE, index=False)

# Create requested final files.
monthly_filtered_output.to_csv(MONTHLY_OUTPUT_FILE, index=False)
daily_filtered_output.to_csv(DAILY_OUTPUT_FILE, index=False)

filtered_month_counts = monthly_filtered_ts.groupby("product_id")["order_month"].nunique()
min_months_after_filter = int(filtered_month_counts.min()) if not filtered_month_counts.empty else 0

required_cols = ["class", "abc_class", "is_demand_stable", "is_fast_moving"]
missing_monthly_cols = [col for col in required_cols if col not in monthly_filtered_output.columns]
missing_daily_cols = [col for col in required_cols if col not in daily_filtered_output.columns]

print("Eligible product_ids:", len(eligible_product_ids))
print("Filtered monthly shape:", monthly_filtered_ts.shape)
print("Filtered daily shape:", daily_filtered_ts.shape)
print("Minimum months per product after filter:", min_months_after_filter)
print("Missing required columns in monthly output:", missing_monthly_cols)
print("Missing required columns in daily output:", missing_daily_cols)
print("Updated:", MONTHLY_TS_FILE)
print("Updated:", DAILY_TS_FILE)
print("Saved:", MONTHLY_OUTPUT_FILE)
print("Saved:", DAILY_OUTPUT_FILE)

display(monthly_filtered_output.head())
display(daily_filtered_output.head())

Eligible product_ids: 409
Filtered monthly shape: (3102, 10)
Filtered daily shape: (6416, 15)
Minimum months per product after filter: 4
Missing required columns in monthly output: []
Missing required columns in daily output: []
Updated: product_monthly_sales_timeseries.csv
Updated: product_daily_sales_timeseries.csv
Saved: product_monthly_sales.csv
Saved: product_daily_sales.csv


,product_id,order_month,quantity_sold,unique_orders,gross_item_sales,gross_freight,total_sales_value,avg_unit_price,product_category_name,product_category_name_english,abc_class,class,is_demand_stable,is_fast_moving
0,005030ef108f58b46b78116f754d8d38,2017-10-01,1,1,13.99,15.09,29.08,13.99,perfumaria,perfumery,C,SY,0,0
1,005030ef108f58b46b78116f754d8d38,2017-11-01,4,4,55.96,43.76,99.72,13.99,perfumaria,perfumery,C,SY,0,0
2,005030ef108f58b46b78116f754d8d38,2018-01-01,1,1,13.99,7.78,21.77,13.99,perfumaria,perfumery,C,SY,0,0
3,005030ef108f58b46b78116f754d8d38,2018-02-01,3,3,41.97,43.03,85.00,13.99,perfumaria,perfumery,C,SY,0,0
4,005030ef108f58b46b78116f754d8d38,2018-04-01,1,1,13.99,7.39,21.38,13.99,perfumaria,perfumery,C,SY,0,0


,product_id,order_date,quantity_sold,unique_orders,gross_item_sales,gross_freight,total_sales_value,avg_unit_price,product_category_name,product_category_name_english,order_month,abc_class,class,is_demand_stable,is_fast_moving
0,005030ef108f58b46b78116f754d8d38,2017-10-20,1,1,13.99,15.09,29.08,13.99,perfumaria,perfumery,2017-10-01,C,SY,0,0
1,005030ef108f58b46b78116f754d8d38,2017-11-14,1,1,13.99,7.78,21.77,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0
2,005030ef108f58b46b78116f754d8d38,2017-11-16,1,1,13.99,7.78,21.77,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0
3,005030ef108f58b46b78116f754d8d38,2017-11-18,1,1,13.99,14.10,28.09,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0
4,005030ef108f58b46b78116f754d8d38,2017-11-24,1,1,13.99,14.10,28.09,13.99,perfumaria,perfumery,2017-11-01,C,SY,0,0
